In [1]:
!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [3]:
df = pd.read_csv(
    "/content/processed_feedback.csv"
)

df.head()

,feedback,processed_text,sentiment,category
0,"Honestly, i would like to schedule deliveries ...",honestly would like schedule delivery need ass...,neutral,feature_request
1,"For some reason, the website response time is ...",reason website response time poor fix,negative,performance
2,I am having trouble because the application is...,trouble application lagging badly please look,negative,performance
3,"Honestly, the payment page shows an error I ne...",honestly payment page show error need assistance,negative,payment
4,I noticed that the application interface is ea...,noticed application interface easy use please ...,negative,ui


In [19]:
df = df[
    [
        "processed_text",
        "sentiment"
    ]
]

df.head()

# This cell (ofSsLm_0nkJH) is not related to the RuntimeError in cell 18 (sTc-adiZr_Ox).
# The error explanation and fix are detailed in the agent's response above.

,processed_text,sentiment
0,honestly would like schedule delivery need ass...,neutral
1,reason website response time poor fix,negative
2,trouble application lagging badly please look,negative
3,honestly payment page show error need assistance,negative
4,noticed application interface easy use please ...,negative


In [5]:
label_mapping = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

df["label"] = df["sentiment"].map(
    label_mapping
)

df.head()

,processed_text,sentiment,label
0,honestly would like schedule delivery need ass...,neutral,1
1,reason website response time poor fix,negative,0
2,trouble application lagging badly please look,negative,0
3,honestly payment page show error need assistance,negative,0
4,noticed application interface easy use please ...,negative,0


In [6]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [7]:
train_dataset = Dataset.from_pandas(
    train_df[
        ["processed_text", "label"]
    ]
)

test_dataset = Dataset.from_pandas(
    test_df[
        ["processed_text", "label"]
    ]
)

In [8]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
def tokenize_function(examples):

    return tokenizer(
        examples["processed_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/5600 [00:00<?, ? examples/s]

Map:   0%|          | 0/1400 [00:00<?, ? examples/s]

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
from sklearn.metrics import accuracy_score


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy
    }

In [12]:
training_args = TrainingArguments(
    output_dir="./transformer_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,

    report_to="none"
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.638781,0.536309,0.767857
2,0.563417,0.530321,0.767143


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1400, training_loss=0.5919088963099889, metrics={'train_runtime': 163.6867, 'train_samples_per_second': 68.423, 'train_steps_per_second': 8.553, 'total_flos': 370915330867200.0, 'train_loss': 0.5919088963099889, 'epoch': 2.0})

In [15]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy
0.563417,0.530321,2,0.767143


{'eval_loss': 0.5303214192390442, 'eval_accuracy': 0.7671428571428571}


In [21]:
def predict_sentiment_transformer(
    feedback
):

    inputs = tokenizer(
        feedback,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model(**inputs)

    logits = outputs.logits

    prediction = torch.argmax(
        logits,
        dim=1
    ).item()

    reverse_mapping = {
        0: "negative",
        1: "neutral",
        2: "positive"
    }

    return reverse_mapping[prediction]

In [22]:
import torch

In [25]:
feedback = (
    "The payment failed and "
    "I am very disappointed."
)

result = predict_sentiment_transformer(
    feedback
)

print(result)

negative
